# **Purpose**

**Track A — Similarity-based baseline (Word/Sentence Embeddings).**

This notebook produces distractors with **no fine-tuned model**: it builds a
pool of candidate phrases (correct answers pulled from RACE train), embeds
them with a sentence encoder, and for each test item retrieves the
**nearest-neighbor phrase** to the correct answer that isn't the correct
answer itself. This is the embedding-based similarity approach described in
the literature review (Section 2.4) — the phrase-level counterpart to
classic single-word GloVe/Word2Vec nearest-neighbor substitution, chosen
because RACE answers are usually full phrases rather than single words.

It scores on the **exact same fixed RACE test split** as the Track B
evaluation notebooks (same `TEST_SIZE`/`SEED`), and saves its predictions to
a JSON file that the matching evaluation notebook consumes.

**Runtime:** CPU works but GPU speeds up embedding the candidate pool.
Internet must be ON (to load RACE and download the sentence encoder).

## **Install dependencies**

In [1]:
!pip install -qU datasets sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 23.5 MB/s eta 0:00:00


## **Load the API Keys and Tokens**

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Imports**

In [3]:
import json
import random

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util as st_util

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


## **Config**

`TEST_SIZE`/`SEED` must match the Track B evaluation notebooks exactly.
`POOL_SIZE` controls how many distinct correct-answer phrases (from RACE
train) form the candidate distractor pool.

In [4]:
TEST_SIZE = 1500          # keep identical to Track B evaluation notebooks
SEED = 42                  # keep identical to Track B evaluation notebooks
POOL_SIZE = 20000          # candidate pool size (distinct correct-answer phrases)
TOP_K = 10                 # neighbors considered before filtering
MAX_SIMILARITY = 0.95      # candidates above this are "too similar" (risk of being a valid answer)

OUTPUT_JSON = "/kaggle/working/embedding_baseline_predictions.json"

## **Load the fixed test split**

RACE gives 3 gold distractors per question, so we first expand the full test
set into individual (context, question, correct answer) → gold-distractor
pairs, then shuffle those pairs with a fixed seed and take the first
`TEST_SIZE`. **Keep `TEST_SIZE`/`SEED` identical to the Track B (T5/BART/Flan-T5)
evaluation notebooks** — that's what makes this baseline directly comparable
to the fine-tuned models.

In [5]:
LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

raw = load_dataset("race", "all")
test_examples = raw["test"]

flat_contexts, flat_questions, flat_answers, flat_distractors = [], [], [], []
for ex in test_examples:
    answer_letter = ex["answer"]
    if answer_letter not in LETTER_TO_IDX:
        continue
    correct_idx = LETTER_TO_IDX[answer_letter]
    options = ex["options"]
    if correct_idx >= len(options):
        continue
    correct_answer = options[correct_idx]

    for i, option in enumerate(options):
        if i == correct_idx or not option.strip():
            continue
        flat_contexts.append(ex["article"])
        flat_questions.append(ex["question"])
        flat_answers.append(correct_answer)
        flat_distractors.append(option)

print(f"Total (context, question, answer) -> gold distractor pairs available: {len(flat_distractors)}")

indices = list(range(len(flat_distractors)))
random.Random(SEED).shuffle(indices)
indices = indices[:TEST_SIZE]

contexts = [flat_contexts[i] for i in indices]
questions = [flat_questions[i] for i in indices]
correct_answers = [flat_answers[i] for i in indices]
gold_distractors = [flat_distractors[i] for i in indices]

print(f"Evaluating on {len(gold_distractors)} pairs")

README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Total (context, question, answer) -> gold distractor pairs available: 14802
Evaluating on 1500 pairs


## **Build the candidate distractor pool**

We reuse correct answers from RACE **train** (a large, diverse bank of
plausible option-length phrases) as the pool to retrieve from, deduplicated
and capped at `POOL_SIZE`.

In [6]:
LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

def collect_correct_answers(split, cap=None):
    answers = []
    for ex in split:
        letter = ex["answer"]
        if letter not in LETTER_TO_IDX:
            continue
        idx = LETTER_TO_IDX[letter]
        options = ex["options"]
        if idx >= len(options):
            continue
        ans = options[idx].strip()
        if ans:
            answers.append(ans)
        if cap and len(answers) >= cap:
            break
    return answers

train_split = raw["train"]
pool_answers = collect_correct_answers(train_split, cap=POOL_SIZE)
pool_answers = list(dict.fromkeys(pool_answers))  # dedupe, preserve order

print(f"Candidate pool size: {len(pool_answers)}")

Candidate pool size: 19101


## **Embed the pool and the test-set correct answers**

In [7]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

pool_embeddings = embedder.encode(
    pool_answers, convert_to_tensor=True, show_progress_bar=True, batch_size=64
)
answer_embeddings = embedder.encode(
    correct_answers, convert_to_tensor=True, show_progress_bar=True, batch_size=64
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

## **Retrieve the nearest non-identical, not-too-similar neighbor**

For each correct answer, we look at its `TOP_K` nearest neighbors in the
pool and pick the first one that (a) isn't the correct answer itself and
(b) isn't above `MAX_SIMILARITY` (which would risk it being a second valid
answer). If every neighbor is above that threshold, we fall back to the
closest non-identical one anyway, so every item still gets a prediction.

In [8]:
hits = st_util.semantic_search(answer_embeddings, pool_embeddings, top_k=TOP_K)

predictions = []
fallback_count = 0

for i, hit_list in enumerate(hits):
    correct = correct_answers[i]
    chosen = ""
    used_fallback = False

    for hit in hit_list:
        candidate = pool_answers[hit["corpus_id"]]
        if candidate.strip().lower() == correct.strip().lower():
            continue
        if hit["score"] > MAX_SIMILARITY:
            continue
        chosen = candidate
        break

    if not chosen:
        for hit in hit_list:
            candidate = pool_answers[hit["corpus_id"]]
            if candidate.strip().lower() != correct.strip().lower():
                chosen = candidate
                used_fallback = True
                break

    if used_fallback:
        fallback_count += 1
    predictions.append(chosen)

print(f"Used the >{MAX_SIMILARITY} fallback for {fallback_count}/{len(correct_answers)} items")

Used the >0.95 fallback for 0/1500 items


## **Save predictions for the evaluation notebook**

In [9]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(
        {
            "method": "embedding_baseline",
            "test_size": TEST_SIZE,
            "seed": SEED,
            "pool_size": len(pool_answers),
            "fallback_count": fallback_count,
            "predictions": predictions,
            "references": gold_distractors,
            "correct_answers": correct_answers,
            "questions": questions,
        },
        f,
        indent=2,
    )

print(f"Saved {len(predictions)} predictions to {OUTPUT_JSON}")

Saved 1500 predictions to /kaggle/working/embedding_baseline_predictions.json


## **Inspect sample generations**

In [10]:
for correct, gold, pred in list(zip(correct_answers, gold_distractors, predictions))[:10]:
    print(f"Correct answer:       {correct}")
    print(f"Gold distractor:      {gold}")
    print(f"Generated distractor: {pred if pred else '(no candidate found)'}")
    print("-" * 80)

Correct answer:       Working or talking with students.
Gold distractor:      Having a basketball game.
Generated distractor: helps students stay in contact with others
--------------------------------------------------------------------------------
Correct answer:       we must know who we are
Gold distractor:      we should know what we do
Generated distractor: everyone has to figure out who we are and why we are here
--------------------------------------------------------------------------------
Correct answer:       the high quality education and research and the wide range of courses
Gold distractor:      the convenient traffic
Generated distractor: is built on important courses and the results of recent studies
--------------------------------------------------------------------------------
Correct answer:       they watch TV late
Gold distractor:      they play the computer games late into the night
Generated distractor: they watch TV too often
---------------------------------

## **Next steps**

- Run `evaluate_embedding_baseline_race_distractor.ipynb` to score these
  predictions with the same metric suite used for T5-base/BART-base/Flan-T5-base.
- Try a larger/smaller `POOL_SIZE` or a different `MAX_SIMILARITY` threshold
  and see how it trades off plausibility vs. validity-risk.
- Try building the pool from options across ALL RACE questions (correct +
  incorrect) rather than only correct answers, for more variety.